## Exploratory Data Analysis & Data Cleansing

In this phase, we performed an initial data quality assessment on the Formula 1 dataset (~518k records, 75 columns). 

Key engineering steps executed:
- **Ingestion & Missing Values Handling:** Identified hidden nulls stored as string placeholders (`\N`) across multiple columns and parsed them into standard `NaN` values directly during ingestion (`na_values=r'\N'`).
- **Data Sanitization:** Removed redundant index column (`Unnamed: 0`) generated from previous exports.
- **Type Conversions:** Cast key numerical attributes (`position`, `milliseconds`, `fastestLap`, `fastestLapSpeed`) from generic `object` types to continuous/discrete numeric types (`float64`/`int64`) for aggregation and analytical suitability.
- **Data Validation:** Verified primary business key integrity (`resultId`, `raceId`, `driverId`, `constructorId`), ensured zero duplicate records, and validated numeric ranges against domain-specific logical boundaries (e.g., F1 speed bounds).

In [1]:
import pandas as pd

df = pd.read_csv('../data/dataEngineeringDataset.csv', na_values=r'\N', low_memory=False)

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print("Dataset shape after handling missing values:", df.shape)

Dataset shape after handling missing values: (518417, 75)


In [2]:

pd.set_option('display.max_rows', None)

pd.DataFrame({
    'Column': df.columns,
    'Non-Null Count': df.notnull().sum().values,
    'Dtype': df.dtypes.values
})

,Column,Non-Null Count,Dtype
0,resultId,518417,int64
1,raceId,518417,int64
2,driverId,518417,int64
3,constructorId,518417,int64
4,number,518417,int64
5,grid,518417,int64
6,position,490703,float64
7,positionText,518417,str
8,positionOrder,518417,int64
9,points,518417,float64


In [3]:
missing_summary = df.isnull().sum()

print("Total missing values across dataset:", missing_summary.sum())
print("\nTop columns with missing values:")
print(missing_summary[missing_summary > 0].sort_values(ascending=False).head(15))

Total missing values across dataset: 4906586

Top columns with missing values:
sprint_time       503851
sprint_date       495270
fp3_time          458901
fp1_time          444335
quali_time        444335
fp2_time          444335
fp3_date          419912
quali_date        396765
fp1_date          396765
fp2_date          396765
milliseconds      219863
time              219863
number_drivers     32608
position           27714
alt                 4191
dtype: int64


In [4]:
numeric_cols = ['milliseconds', 'position', 'fastestLap', 'fastestLapSpeed']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols].dtypes

milliseconds       float64
position           float64
fastestLap         float64
fastestLapSpeed    float64
dtype: object

In [5]:
df[['position', 'milliseconds', 'fastestLapSpeed']].describe()

,position,milliseconds,fastestLapSpeed
count,490703.000000,2.985540e+05,518046.000000
mean,9.627110,6.154901e+06,202.071695
std,5.302606,1.145127e+06,20.230435
min,1.000000,4.526665e+06,100.615000
25%,5.000000,5.506608e+06,191.302000
50%,10.000000,5.836977e+06,202.808000
75%,14.000000,6.326965e+06,214.436000
max,23.000000,1.101210e+07,255.014000


In [6]:
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [7]:
print("Min/Max Fastest Lap Speed:", df['fastestLapSpeed'].min(), df['fastestLapSpeed'].max())
print("Negative or zero milliseconds count:", (df['milliseconds'] <= 0).sum())

Min/Max Fastest Lap Speed: 100.615 255.014
Negative or zero milliseconds count: 0


In [8]:
if 'constructorName' in df.columns:
    print("Unique constructors count:", df['constructorName'].nunique())

In [9]:
key_cols = ['resultId', 'raceId', 'driverId', 'constructorId']
print("Nulls in key identifier columns:")
print(df[key_cols].isnull().sum())

Nulls in key identifier columns:
resultId         0
raceId           0
driverId         0
constructorId    0
dtype: int64


In [10]:
pd.set_option('display.max_columns', None)

df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,year,round,circuitId,name_x,date,time_races,url_x,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,circuitRef,name_y,location,country,lat,lng,alt,url_y,driverRef,number_drivers,code,forename,surname,dob,nationality,url,constructorRef,name,nationality_constructors,url_constructors,lap,position_laptimes,time_laptimes,milliseconds_laptimes,stop,lap_pitstops,time_pitstops,duration,milliseconds_pitstops,driverStandingsId,points_driverstandings,position_driverstandings,positionText_driverstandings,wins,constructorStandingsId,points_constructorstandings,position_constructorstandings,positionText_constructorstandings,wins_constructorstandings,status
0,21232,860,18,1,3,2,1.0,1,1,25.0,58,1:34:09.565,5649565.0,56.0,1,1:29.187,214.053,1,2012,1,1,Australian Grand Prix,2012-03-18,06:00:00,http://en.wikipedia.org/wiki/2012_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,button,22.0,BUT,Jenson,Button,1980-01-19,British,http://en.wikipedia.org/wiki/Jenson_Button,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,1,1,1:39.264,99264,1,16,17:28:30,23.275,23275,64976,25.0,1,1,1,24932,40.0,1,1,1,Finished
1,21232,860,18,1,3,2,1.0,1,1,25.0,58,1:34:09.565,5649565.0,56.0,1,1:29.187,214.053,1,2012,1,1,Australian Grand Prix,2012-03-18,06:00:00,http://en.wikipedia.org/wiki/2012_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,button,22.0,BUT,Jenson,Button,1980-01-19,British,http://en.wikipedia.org/wiki/Jenson_Button,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,1,1,1:39.264,99264,2,36,17:59:44,22.837,22837,64976,25.0,1,1,1,24932,40.0,1,1,1,Finished
2,21232,860,18,1,3,2,1.0,1,1,25.0,58,1:34:09.565,5649565.0,56.0,1,1:29.187,214.053,1,2012,1,1,Australian Grand Prix,2012-03-18,06:00:00,http://en.wikipedia.org/wiki/2012_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,button,22.0,BUT,Jenson,Button,1980-01-19,British,http://en.wikipedia.org/wiki/Jenson_Button,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,2,1,1:33.414,93414,1,16,17:28:30,23.275,23275,64976,25.0,1,1,1,24932,40.0,1,1,1,Finished
3,21232,860,18,1,3,2,1.0,1,1,25.0,58,1:34:09.565,5649565.0,56.0,1,1:29.187,214.053,1,2012,1,1,Australian Grand Prix,2012-03-18,06:00:00,http://en.wikipedia.org/wiki/2012_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,button,22.0,BUT,Jenson,Button,1980-01-19,British,http://en.wikipedia.org/wiki/Jenson_Button,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,2,1,1:33.414,93414,2,36,17:59:44,22.837,22837,64976,25.0,1,1,1,24932,40.0,1,1,1,Finished
4,21232,860,18,1,3,2,1.0,1,1,25.0,58,1:34:09.565,5649565.0,56.0,1,1:29.187,214.053,1,2012,1,1,Australian Grand Prix,2012-03-18,06:00:00,http://en.wikipedia.org/wiki/2012_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,button,22.0,BUT,Jenson,Button,1980-01-19,British,http://en.wikipedia.org/wiki/Jenson_Button,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,3,1,1:33.350,93350,1,16,17:28:30,23.275,23275,64976,25.0,1,1,1,24932,40.0,1,1,1,Finished
